In [ ]:
"""
Fairness Analysis for AI Detection Counts Across Protected Attribute Groups
===========================================================================
This script performs a transparent, step-by-step fairness analysis to detect
whether detection counts differ across protected attribute groups (gender, race, age).
"""

import pandas as pd
import warnings
import os

warnings.filterwarnings("ignore")

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
# In Jupyter, __file__ is not available; use the notebook's working directory instead
_HERE = os.path.abspath("")                                  # .../bias-report/utils
_ROOT = os.path.dirname(_HERE)                               # .../bias-report

INPUT_CSV  = os.path.join(_ROOT, "bias-report/data", "combined.csv")
OUTPUT_DIR = os.path.join(_ROOT, "bias-report/outputs")
# SUMMARY_CSV = os.path.join(OUTPUT_DIR, "fairness_summary.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

ALPHA = 0.05  # significance threshold for Wilcoxon test

In [ ]:
# Get latest prompts

from utils.get_prompts import get_prompts

prompts = get_prompts()

prompts

In [ ]:
# Get counterfactual snippets to test

from utils.get_snippets import get_snippets

snippets = get_snippets()

In [ ]:
from utils.run_dispro_analysis import run_prompts_over_snippets, postprocess_analysis_results

for i in range(3):

    # Step 1: Run prompts and get results

    df = run_prompts_over_snippets(prompts=prompts, snippets_df=snippets, print_responses=True, run_id=i)

    df

    # # Step 2: Post-process

    CSV_NAME=f"data/dispro_analysis/simplified_joined_results{i}.csv"

    simplified_joined_results = postprocess_analysis_results(
        results_df= df,
        snippets_df= snippets,
        save_to_csv=CSV_NAME
    )

In [ ]:
import glob

# Get all CSVs in the folder
csv_files = glob.glob("data/dispro_analysis/*.csv")  # or "path/*.csv"

# Combine
df = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)

df.to_csv("data/combined.csv", index=False)

In [ ]:

df = pd.read_csv("data/combined.csv")

df = df.rename(columns={"prompt_name": "prompt"})

df.to_csv("data/combined.csv", index=False)


In [ ]:
# ─────────────────────────────────────────────
# STEP 1 — Load and Clean the Dataset
# ─────────────────────────────────────────────

from utils.bias_analysis import load_and_clean

df=load_and_clean(INPUT_CSV)

df.head()

In [ ]:
# ─────────────────────────────────────────────
# STEP 1b — Aggregate Across Runs and Prompts
# ─────────────────────────────────────────────

from utils.bias_analysis import aggregate_runs

per_prompt_agg, combined_agg = aggregate_runs(df)
per_prompt_agg.head()

In [ ]:
# ─────────────────────────────────────────────
# STEP 2 — Pair Observations by Snippet and Attribute
# ─────────────────────────────────────────────

from utils.bias_analysis import build_pairs

# Combined across all prompts
gender_pairs = build_pairs(combined_agg, "gender")
age_pairs    = build_pairs(combined_agg, "age")
race_pairs   = build_pairs(combined_agg, "race")


In [ ]:
gender_pairs

In [ ]:
# ─────────────────────────────────────────────
# STEP 6 — Build Pairs Per Prompt
# ─────────────────────────────────────────────

from utils.bias_analysis import build_pairs_by_prompt

gender_pairs_by_prompt = build_pairs_by_prompt(per_prompt_agg, "gender")
age_pairs_by_prompt    = build_pairs_by_prompt(per_prompt_agg, "age")
race_pairs_by_prompt   = build_pairs_by_prompt(per_prompt_agg, "race")

In [ ]:
# ─────────────────────────────────────────────
# STEP 7 — Compare Differences Across Prompts
# ─────────────────────────────────────────────

from utils.bias_analysis import compare_prompts


gender_prompt_comparison = compare_prompts(gender_pairs_by_prompt, "gender")
age_prompt_comparison    = compare_prompts(age_pairs_by_prompt,    "age")
race_prompt_comparison   = compare_prompts(race_pairs_by_prompt,   "race")

gender_prompt_comparison

In [ ]:
from utils.report_generation import generate_bias_report

generate_bias_report(
    df                       = df,
    gender_pairs             = gender_pairs,
    age_pairs                = age_pairs,
    race_pairs               = race_pairs,
    gender_prompt_comparison = gender_prompt_comparison,
    age_prompt_comparison    = age_prompt_comparison,
    race_prompt_comparison   = race_prompt_comparison,
    output_dir = OUTPUT_DIR
)